In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
#
# A program that takes a csv and trains models on it. Streamlined model selection.
# ==============================================================================

import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
# import pandas_profiling
import os

# Fit an xgboost model

# TabNet
# from fast_tabnet.core import *


In [ ]:
# Project Variables
# ===================================================================================================
PROJECT_NAME = "superstore"
VARIABLE_FILES = False
# Maximum amount of rows to take

## -- STEFANOS -- Remove sample

# SAMPLE_COUNT = 4000
FASTAI_LEARNING_RATE = 1e-1
AUTO_ADJUST_LEARNING_RATE = False
# Set to True automatically infer if variables are categorical or continuous
ENABLE_BREAKPOINT = True
# When trying to declare a column a continuous variable, if it fails, convert it to a categorical variable
CONVERT_TO_CAT = False
REGRESSOR = True
SEP_DOLLAR = False
SEP_COMMA = True
SHUFFLE_DATA = True

In [ ]:
### cell 0 ###

PARAM_DIR = "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/input/superstore"
df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/input/superstore/SampleSuperstore.csv"
)
factor = 10
df = pd.concat([df] * factor)
df.info()

In [ ]:
### cell 1 ###

if SEP_DOLLAR:
    # For every column in df_copy, if it contains a “$”, remove “$” and commas, then cast to float on GPU
    for col in df_copy.columns:
        s = df[col].astype("str")
        if s.str.contains("$", regex=False).any():
            cleaned = s.str.replace("$", "", regex=False).str.replace(
                ",", "", regex=False
            )
            df[col + "_no_dollar"] = cleaned.astype("float64")

if SEP_COMMA:
    # For every column, if it contains “%” or comma, remove them, then cast to float on GPU
    for col in df.columns:
        s = df[col].astype("str")
        if (
            s.str.contains("%", regex=False).any()
            or s.str.contains(",", regex=False).any()
        ):
            cleaned = s.str.replace("%", "", regex=False).str.replace(
                ",", "", regex=False
            )
            df[col + "_processed"] = cleaned.astype("float64")

In [ ]:
### cell 2 ###

df

In [ ]:
### cell 3 ###

df.isna().sum()

In [ ]:
### cell 4 ###

_ = df.select_dtypes(include=["number"]).corr()

In [ ]:
### cell 5 ###

df.head()

In [ ]:
### cell 6 ###

df.describe().T

In [ ]:
### cell 7 ###

df.columns

In [ ]:
### cell 8 ###

target = ""
target_str = ""
targets = []

# Loop through every possible target column (Continuous)
for col in reversed(df.columns[1:]):
    try:
        # Try to convert this candidate to numeric
        df[col] = pd.to_numeric(df[col], errors="coerce")
        target = col
        target_str = target.replace("/", "-")
    except:
        continue
    print(f"Target Variable: {target}")

    # Ensure PARAM_DIR and its config files exist
    os.makedirs(PARAM_DIR, exist_ok=True)
    for fname in ["cats.txt", "conts.txt", "cols_to_delete.txt"]:
        path = os.path.join(PARAM_DIR, fname)
        open(path, "a").close()

    # Drop duplicate rows
    df = df.drop_duplicates()
    # if SHUFFLE_DATA:
    #     df = df.sample(frac=1).reset_index(drop=True)

    # Cast any boolean columns to uint8 (workaround for fastai/pytorch)
    for n in df.columns:
        if pd.api.types.is_bool_dtype(df[n]):
            df[n] = df[n].astype("uint8")

    # Drop any columns listed in cols_to_delete.txt
    with open(os.path.join(PARAM_DIR, "cols_to_delete.txt"), "r") as f:
        cols_to_delete = f.read().splitlines()
    for c in cols_to_delete:
        if c in df.columns:
            df.drop(columns=c, inplace=True)

    # Fill remaining NaNs with zero
    try:
        df = df.fillna(0)
    except:
        pass

    # Auto-detect categorical vs continuous (unique/count ratio)
    ratio = df.nunique() / df.count()
    cats = [v for v, r in ratio.items() if r < 0.05]
    conts = [v for v, r in ratio.items() if r >= 0.05]

    # Exclude the target variable from feature lists
    if target in cats:
        cats.remove(target)
    if target in conts:
        conts.remove(target)

    # Ensure target is numeric
    try:
        df[target] = pd.to_numeric(df[target], errors="coerce")
    except:
        pass

    # Override cats/conts from files if configured
    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, "cats.txt"), "r") as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, "conts.txt"), "r") as f:
            conts = f.read().splitlines()

    # Convert continuous features to numeric
    for var in conts:
        try:
            df[var] = pd.to_numeric(df[var], errors="coerce")
        except:
            print(f"Could not convert {var} to float.")

    # Optional breakpoint: re-validate continuous conversions
    if ENABLE_BREAKPOINT:
        cont_list = list(conts)
        for var in cont_list:
            try:
                df[var] = pd.to_numeric(df[var], errors="coerce")
            except:
                print(f"Could not convert {var} to float.")
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)

In [ ]:
# out_dir = f'./{PROJECT_NAME}'
# xgb_feature_importance_csvs = []

# for file in os.listdir(out_dir):
#     if 'xgb_feature_importance' in file and '.csv' in file:
#         xgb_feature_importance_csvs.append(pd.read_csv(os.path.join(out_dir, file)))

# xgb_feature_importance = pd.concat(xgb_feature_importance_csvs,axis=0)
# xgb_feature_importance.rename(columns={'Unnamed: 0': 'feature'}, inplace=True)
# print(xgb_feature_importance.head())
# xgb_feature_importance.groupby('feature')['importance'].mean().sort_values(ascending=False).plot(kind='bar', title='XGBoost Overall Feature Importance', figsize=(20, 10))

In [ ]:
### cell 9 ###

df.isna().sum()